# 02 – Scope Selection

**Purpose:** Choose a scope that fairly tests demand sensing and fits a 16 GB laptop.

**Inputs:** `sales`, `items`, `stores`
**Outputs:** table `scope_item_profile`

**Scope locked:**
- Dates: weekly plan 2014-04-01 → 2017-08-15; daily sensing 2016-01-01 → 2017-08-15
- Families: Beverages (non-perishable, weather-sensitive), Dairy, Meats (perishable, higher promo)
- Stores: 13 across 9 cities, highlands and coast, all 5 store types, full history
- Items: 813 active items with enough history

In [1]:
import duckdb
con = duckdb.connect('../data/favorita.duckdb')   # creates the file on first run
con.sql("SET memory_limit = '4GB'")
con.sql("SET preserve_insertion_order = false")
con.sql("SET temp_directory = '../data/tmp'")

## Model Product Families

** Check items and sales table before proceeding

** For each family: the number of distinct items sold, total units sold, the family’s % share of total units, the % of its items that are perishable, and the % of its sales rows that were on promotion. Sort by total units, highest first.


In [4]:
con.sql("""SELECT * FROM sales LIMIT 2""").df()

,date,store_nbr,item_nbr,unit_sales,is_on_promotion
0,2013-01-05,1,578097,9.0,<NA>
1,2013-01-05,1,578467,2.0,<NA>


In [5]:
con.sql("""SELECT * FROM items LIMIT 2""").df()

,item_nbr,family,class,is_perishable
0,96995,GROCERY I,1093,False
1,99197,GROCERY I,1067,False


In [36]:
con.sql("""WITH sales_items AS (
            SELECT 
            s.date,
            s.store_nbr,
            s.item_nbr,
            s.unit_sales, 
            s.is_on_promotion,
            i.family,
            i.class,
            i.is_perishable
            FROM sales s
            LEFT JOIN items i
                ON s.item_nbr = i.item_nbr)
            SELECT 
            family,
            COUNT(DISTINCT item_nbr) AS dist_items_sold,
            SUM(unit_sales) AS total_units_sale,
            SUM(unit_sales)*100.0/SUM(SUM(unit_sales)) OVER() AS pct_unit_sale_family,
            COUNT(DISTINCT CASE WHEN is_perishable = 'True' THEN item_nbr END)*100.0/COUNT(DISTINCT item_nbr) AS pct_items_perishable,
            SUM(CASE WHEN is_on_promotion = 'True' THEN 1 ELSE 0 END)*100.0/COUNT(*) AS pct_rows_on_promo
            FROM sales_items
            WHERE date BETWEEN '2014-04-01' AND '2017-08-15'
            GROUP BY family
            ORDER By total_units_sale DESC""").df()
            
            

,family,dist_items_sold,total_units_sale,pct_unit_sale_family,pct_items_perishable,pct_rows_on_promo
0,GROCERY I,1290,2.699468e+08,30.610492,0.0,5.213157
1,BEVERAGES,604,1.846716e+08,20.940753,0.0,5.959875
2,PRODUCE,306,1.160293e+08,13.157093,100.0,16.400218
3,CLEANING,446,7.520832e+07,8.528213,0.0,4.953070
4,DAIRY,242,5.341558e+07,6.057035,100.0,9.602717
5,BREAD/BAKERY,134,3.329668e+07,3.775661,100.0,8.857784
6,POULTRY,54,2.559246e+07,2.902045,100.0,15.192237
7,MEATS,84,2.310294e+07,2.619747,100.0,16.470029
8,PERSONAL CARE,153,1.968770e+07,2.232478,0.0,6.468839
9,DELI,91,1.908097e+07,2.163678,100.0,18.218292


**Finding (2014-04-01 → 2017-08-15):** 33 families. The top 5 (Grocery I, Beverages, Produce, Cleaning, Dairy) carry ~80% of units; the bottom ~15 families carry under 1% combined and are too sparse to forecast well. Perishability is set at family level (every family is 0% or 100% perishable). Fresh families run 15–20% promo rows vs ~5% for dry grocery and beverages. Table size is driven by item count, not volume: Grocery I alone would be ~40M daily rows across all stores.

**Decision – families in scope:**
| Family | Perishable | Units share | Promo rows | Items | Why |
|---|---|---|---|---|---|
| BEVERAGES | No | 20.9% | 6.0% | 604 | High volume, non-perishable, most weather-sensitive |
| DAIRY | Yes | 6.1% | 9.6% | 242 | High volume, perishable, medium promo |
| MEATS | Yes | 2.6% | 16.5% | 84 | Perishable, high promo; sensing expected to matter most |

~930 items covering both contrasts the FVA analysis needs: **perishable vs non-perishable** and **low vs high promo intensity**.

**Not in scope:** Grocery I (largest family, but 1,290 items would double the table size, demand is stable and low-promo, and no price data exists to model price sensitivity). Noted as a next step.

**Limitation (README):** Favorita has no price data; only a yes/no promo flag is available, so price elasticity is not modelled.

## Model Stores for Demand Sensing

** For each store: store_nbr, city, state, store_type, its first sales date in sales, and its total units sold in Beverages, Dairy and Meats between 2014-04-01 and 2017-08-15. Sort by city, then total units highest first.

** Check stores table 

In [38]:
con.sql("""SELECT * FROM stores LIMIT 2""").df()

,store_nbr,city,state,store_type,cluster
0,1,Quito,Pichincha,D,13
1,2,Quito,Pichincha,D,13


In [39]:
con.sql("""SELECT * FROM sales LIMIT 2""").df()

,date,store_nbr,item_nbr,unit_sales,is_on_promotion
0,2013-01-01,25,103665,7.0,<NA>
1,2013-01-01,25,105574,1.0,<NA>


In [60]:
con.sql("""SELECT 
            st.store_nbr,
            st.city,
            st.state,
            st.store_type,
            Min(s.date) AS first_sale_date,
            SUM(CASE WHEN i.family IN ('BEVERAGES','DAIRY','MEATS') AND s.date BETWEEN '2014-04-01' AND '2017-08-15'THEN s.unit_sales END) AS total_units_sold,
            FROM sales s
            LEFT JOIN stores st
                ON s.store_nbr = st.store_nbr
            LEFT JOIN items i
                ON s.item_nbr = i.item_nbr
            GROUP BY st.store_nbr, st.city, st.state, st.store_type
            ORDER BY st.city,total_units_sold DESC""").df()
            
            

,store_nbr,city,state,store_type,first_sale_date,total_units_sold
0,23,Ambato,Tungurahua,D,2013-01-02,2.961689e+06
1,50,Ambato,Tungurahua,A,2013-01-02,6.655519e+06
2,31,Babahoyo,Los Rios,B,2013-01-02,3.872785e+06
3,11,Cayambe,Pichincha,B,2013-01-02,6.441546e+06
4,42,Cuenca,Azuay,D,2015-08-21,2.647536e+06
5,39,Cuenca,Azuay,B,2013-01-02,4.407647e+06
6,37,Cuenca,Azuay,D,2013-01-02,4.445799e+06
7,27,Daule,Guayas,D,2013-01-02,4.874085e+06
8,54,El Carmen,Manabi,C,2013-01-02,2.239324e+06
9,43,Esmeraldas,Esmeraldas,E,2013-01-02,4.877852e+06


**Finding:** 54 stores across 22 cities; Quito (Sierra) and Guayaquil (Costa) dominate. Seven stores opened after 2014-04-01 (52, 53, 42, 29, 20, 21, 22) and six have unusual transaction gaps (18, 24, 25, 12, 14, 30).

**Decision – stores in scope (13):** Sierra: 44, 3, 17 (Quito), 39 (Cuenca), 50 (Ambato), 38 (Loja), 11 (Cayambe). Costa: 51, 34, 28 (Guayaquil), 40 (Machala), 43 (Esmeraldas), 27 (Daule). Covers 9 cities in both climate zones and all 5 store types; all have full history from 2013 and normal gap levels.

## Scope Step 3 – Which items to model

** Table scope_item_profile with one row per item in Beverages, Dairy and Meats, using only sales from the 13 scoped stores between 2014-04-01 and 2017-08-15. For each item, show: item_nbr, family, is_perishable, its first sale date, its last sale date, the number of scoped stores it sold in, and its total units.

In [75]:
    con.sql("""CREATE OR REPLACE TABLE scope_item_profile AS
                WITH sales_item AS (
                SELECT
                item_nbr, 
                Min(date) AS first_sale_date,
                Max(date) AS last_sale_date,
                SUM(unit_sales) AS tot_unit_sales,
                COUNT(DISTINCT store_nbr) AS stores_in_scope
                FROM sales
                WHERE date BETWEEN '2014-04-01' AND '2017-08-15'
                AND unit_sales >0
                AND store_nbr IN (44, 3, 17, 39, 50, 38, 11, 51, 34, 28, 40, 43, 27)
                GROUP BY item_nbr)
                SELECT 
                s.item_nbr,
                i.family,
                i.is_perishable,
                s.first_sale_date,
                s.last_sale_date,
                s.tot_unit_sales,
                s.stores_in_scope
                FROM sales_item s
                LEFT JOIN items i
                    ON s.item_nbr = i.item_nbr
                WHERE i.family IN ('BEVERAGES','DAIRY','MEATS')""")

In [76]:
con.sql("""SELECT * FROM scope_item_profile""").df()

,item_nbr,family,is_perishable,first_sale_date,last_sale_date,tot_unit_sales,stores_in_scope
0,414305,BEVERAGES,False,2014-04-01,2017-03-26,75001.0,13
1,468265,BEVERAGES,False,2014-04-01,2017-08-15,106787.0,13
2,556698,BEVERAGES,False,2014-04-01,2017-08-15,68069.0,13
3,568133,BEVERAGES,False,2014-04-01,2017-08-15,45005.0,13
4,716250,BEVERAGES,False,2014-04-01,2017-08-15,232920.0,13
...,...,...,...,...,...,...,...
922,2087567,DAIRY,True,2017-05-13,2017-08-15,11408.0,13
923,2083837,BEVERAGES,False,2017-05-01,2017-08-15,9513.0,13
924,2062477,BEVERAGES,False,2017-05-05,2017-08-15,608.0,8
925,2127114,BEVERAGES,False,2017-07-17,2017-08-15,361.0,5


In [78]:
con.sql("""SELECT
            COUNT(*) AS tot_no_of_itmes,
            SUM(CASE WHEN last_sale_date < '2017-07-01' THEN 1 END) AS last_sale_before_2017_07_01,
            SUM(CASE WHEN first_sale_date >= '2017-01-01' THEN 1 END) AS first_sale_after_2017_01_01
            FROM scope_item_profile""").df()

,tot_no_of_itmes,last_sale_before_2017_07_01,first_sale_after_2017_01_01
0,927,60.0,54.0


In [81]:
con.sql("""SELECT
            COUNT(*) AS tot_no_of_itmes,
            FROM scope_item_profile
            WHERE last_sale_date >= '2017-07-01'
            AND first_sale_date < '2017-01-01' """).df()

,tot_no_of_itmes
0,813


**Finding:** 927 items in Beverages, Dairy and Meats sold in the 13 scoped stores between 2014-04-01 and 2017-08-15. 60 items (6.5%) last sold before 2017-07-01 (likely discontinued) and 54 (5.8%) first sold on or after 2017-01-01 (too new). No item failed both rules.

**Decision – items in scope: 813**, all with first sale before 2017-01-01 and last sale on or after 2017-07-01. New items are parked under new product forecasting; discontinued items are out of scope. The calendar is built per item × store pair from the pair's first sale date. First and last sale dates use units > 0, so returns-only days don't count as sales.

In [2]:
con.close()